In [ ]:
# this is the feature extraction pipeline so we can get the embeddings directly (we can only do inference with this, no fine-tuning)
from transformers import AutoTokenizer, AutoModel

model_name = "Stevenf232/context_fine-tuned-SapBERT1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'
model.to(device)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")
data = dataset["test"]

There are 2 ways we can go about this:
1. Using a [SEP] token
2. using a custom [MENTION] token (this will have to be learned in fine-tuning) - we're not doing this here!

## 1. [SEP] token

In [ ]:
import torch
from tqdm import tqdm

def extract_features(model, tokenizer, text_a, text_b, batch_size=128, device="cuda"):
    model.eval()
    model.to(device)

    all_embeddings = []

    total_batches = (len(text_a) + batch_size - 1) // batch_size

    for i in tqdm(
        range(0, len(text_a), batch_size),
        total=total_batches,
        desc="Encoding",
    ):
        batch_a = text_a[i:i+batch_size]
        batch_b = text_b[i:i+batch_size]


        # when we structure input like these, we get something like:
        # "[CLS] naloxone [SEP] naloxone reverses the antihypertensive effect of clonidine. [SEP] [PAD] [PAD] ..."
        inputs = tokenizer(
            batch_a,
            batch_b,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=128
        )

        inputs.to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        cls = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls.cpu())

    return torch.cat(all_embeddings, dim=0)


In [ ]:
# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window


mention_cls = extract_features(
    model,
    tokenizer,
    [p['mention'] for p in data],
    [get_mt_window(p['mention'], p['mention_text']) for p in data]
)

entity_cls = extract_features(
    model,
    tokenizer,
    [p['entity'] for p in data],
    [p['definition'][:128] for p in data]
)


# Model evaluation

Potential issue - this finds relevance using Cosine Similarity (will it have bias towards fine-tuning on cosineSimilarityLoss vs other loss functions?)

In [ ]:
def evaluate(mention_cls, entity_cls, data):
    print("Processing vectors on GPU (CLS Pooling)...")

    # 1. move to GPU
    mentions_tensor = mention_cls.to('cuda')
    entities_tensor = entity_cls.to('cuda')

    # --- 2. Normalize ---
    # Standardize vector length so Dot Product = Cosine Similarity
    mentions_norm = torch.nn.functional.normalize(mentions_tensor, p=2, dim=1)
    entities_norm = torch.nn.functional.normalize(entities_tensor, p=2, dim=1)

    # --- 3. Matrix Multiplication ---
    # Compute similarity between ALL mentions and ALL entities instantly
    similarity_matrix = torch.mm(mentions_norm, entities_norm.T)

    # --- 4. Find Best Matches ---
    # Returns the index of the highest score for each row
    top_indices = torch.argmax(similarity_matrix, dim=1).cpu().numpy()

    # --- 5. Print Loop ---
    correct_count = 0
    print("\n--- Starting Evaluation ---\n")

    for i, top_idx in enumerate(top_indices):
        # the strange conversion to int from here on out is because the original idx is of type numpy.int64
        top_idx = int(top_idx)
        i = int(i)

        top_match_id = data[top_idx]["id"]
        correct_id = data[i]["id"]

        if top_match_id == correct_id:
            correct_count += 1

        mention_name = data[i]["mention"]
        top_match = data[top_idx]["entity"]
        correct_name = data[i]["entity"]
        passage_text = data[i]["mention_text"]
        definition = data[i]["definition"]
        definition = repr(definition)[1:-1] # remove the \n

        print(f"mention_name: {mention_name}")
        print(f"correct entity name: {correct_name}")
        print(f"entity definition: {definition}")
        print(f"top_match: {top_match}")
        print(f"passage text: {passage_text}")

        print("")

    # --- 6. Statistics ---
    accuracy = correct_count / len(data)
    print(f"total comparisons: {len(data)}")
    print(f"correct comparisons: {correct_count}")
    print(f"accuracy: {accuracy:.4f}")

    return accuracy

In [ ]:
evaluate(mention_cls, entity_cls, data)

## Evaluating Sentence Transformer model


In [ ]:
from sentence_transformers import SentenceTransformer

fine_tuned_model_name = "Stevenf232/fine-tuned-SapBERT4"
model = SentenceTransformer(fine_tuned_model_name)

In [ ]:
from tqdm import tqdm
def encode(model, pairs):
  batch_size=16
  mention_encodings = []
  entity_encodings = []

  for i in tqdm(range(0, len(pairs), batch_size), desc="Extracting features"):
      # encode mentions
      batch = pairs[i:i + batch_size]["mention"]
      encodings = model.encode(batch, convert_to_tensor=True)
      mention_encodings.extend(encodings)

      # encode entities
      batch = pairs[i:i + batch_size]["entity"]
      encodings = model.encode(batch, convert_to_tensor=True)
      entity_encodings.extend(encodings)

  return mention_encodings, entity_encodings

In [ ]:
mention_encodings, entity_encodings = encode(model, data)

In [ ]:
evaluate(mention_encodings, entity_encodings, data)